# PyTorch 入門

Tensor・autograd・学習ループ・Dataset/DataLoader という PyTorch の基本要素を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/deep-learning/02-pytorch-intro`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

## Tensor と autograd

PyTorch の中心は `Tensor`（NumPy 配列に似た多次元配列、GPU 対応）と、
自動微分の **autograd** です。`requires_grad=True` のテンソルに対する計算は記録され、
`backward()` で勾配が自動計算されます。

In [ ]:
import torch

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()
print(x.grad)   # tensor([2., 4., 6.])  ← dy/dx = 2x

## Dataset と DataLoader

データ供給は `Dataset`（1 件の取り出し方）と `DataLoader`（バッチ化・シャッフル）で行います。

In [ ]:
# 先のセルで autograd 例としてスカラー y が定義されたため、表形式学習用の特徴量 X と二値ラベル y を準備します
import numpy as np

rng = np.random.default_rng(42)
X = rng.normal(size=(256, 8)).astype(np.float32)
y = ((X[:, 0] - 0.7 * X[:, 1]) > 0).astype(np.float32)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TableDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

loader = DataLoader(TableDataset(X, y), batch_size=64, shuffle=True)

## 学習ループ

PyTorch では学習ループを自分で書きます。**勾配のゼロ初期化 → 順伝播 → 損失 → 逆伝播 → 更新**が定型です。

In [ ]:
import torch.nn as nn

model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

model.train()
for xb, yb in loader:
    optimizer.zero_grad()          # 勾配を初期化
    pred = model(xb).squeeze(1)    # 順伝播
    loss = loss_fn(pred, yb)       # 損失
    loss.backward()                # 逆伝播
    optimizer.step()               # 更新

In [ ]:
print(f"Final mini-batch training loss: {loss.item():.4f}")

> **Warning:** `optimizer.zero_grad()` を忘れると勾配が蓄積され、学習が壊れます。各ステップの先頭で必ず呼びます。 推論時は `model.eval()` と `torch.no_grad()` を使い、Dropout 等を無効化します。

## 次のステップ

次は、表形式データに DL を適用する方法（MLP・Entity Embedding）を学びます。